# Lab 18-03 · Interrupts on a Real (Emulated) NVIC, and Timing by Analysis

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import re, tempfile
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import fwrun
import irqtime as it

## Parts 1–4 · The firmware on QEMU

*Predict:* the priority bits QEMU implements, the event log (1 = IRQ0 in, 2 = IRQ0 out, 3 = IRQ1, 4 = IRQ2), EXC_RETURN, and CFSR, HFSR and BFAR after a read of 0x5000 0000.

In [2]:
status, out, elf = fwrun.build_and_run(Path(tempfile.mkdtemp(prefix="lab1803_")))
print("exit status", status)
print(out)
rep = {k: int(x, 16) for k, x in re.findall(r"^(\w+) (0x[0-9A-F]{8})$", out, re.M)}
print("event log:", list(rep["ORDER"].to_bytes(4, "big")))
cfsr = rep["CFSR"]
names = {9: "PRECISERR", 15: "BFARVALID", 8: "IBUSERR", 10: "IMPRECISERR", 1: "DACCVIOL", 16: "UNDEFINSTR", 25: "DIVBYZERO"}
print("CFSR bits set:", [names.get(b, b) for b in range(32) if cfsr >> b & 1])

exit status 0
NVIC priority bits: 8
ORDER 0x01030204
EXC_RETURN 0xFFFFFFF9
PRIO_READBACK 0x000000FF
CFSR 0x00008200
HFSR 0x40000000
BFAR 0x50000000
TEST PASSED

event log: [1, 3, 2, 4]
CFSR bits set: ['PRECISERR', 'BFARVALID']


## Part 5 · Response times

*Predict* R_C for Example 18-03.3, and R_C when B's handler grows to 25 µs.

In [3]:
irqs = [it.Irq("A", 5, 50, 0), it.Irq("B", 10, 200, 1), it.Irq("C", 20, 1000, 2)]
for blk in (8, 30):
    print(f"blocking {blk} us: recurrence", {n: it.response_time(irqs, n, blk) for n in "ABC"},
          " simulation", it.simulate(irqs, 3000, blocking_at=0.0, blocking_len=blk))
irqs2 = [it.Irq("A", 5, 50, 0), it.Irq("B", 25, 200, 1), it.Irq("C", 20, 1000, 2)]
print("B = 25 us:", {n: it.response_time(irqs2, n, 8) for n in "ABC"})
# random arrival offsets never beat the critical instant
rng = np.random.default_rng(1803)
worst = 0
for _ in range(200):
    off = {n: float(rng.uniform(0, 50)) for n in "AB"}
    worst = max(worst, it.simulate(irqs, 3000, offsets=off)["C"])
print("worst R_C over 200 random phasings (no blocking):", worst, "<= recurrence without blocking:", it.response_time(irqs, "C", 0))

blocking 8 us: recurrence {'A': 13, 'B': 23, 'C': 43}  simulation {'A': 13.0, 'B': 23.0, 'C': 43.0}
blocking 30 us: recurrence {'A': 35, 'B': 45, 'C': 70}  simulation {'A': 35.0, 'B': 45.0, 'C': 70.0}
B = 25 us: {'A': 13, 'B': 38, 'C': 63}
worst R_C over 200 random phasings (no blocking): 35.0 <= recurrence without blocking: 35


## Part 6 · Timers

*Predict* the highest 10-bit PWM frequency from 84 MHz, and PSC/ARR for 1 s from 72 MHz.

In [4]:
bits = np.arange(6, 17)
plt.semilogy(bits, 84e6 / 2.0 ** bits, "o-")
plt.xlabel("PWM resolution (bits)"); plt.ylabel("max PWM frequency (Hz)"); plt.title("84 MHz timer, edge-aligned"); plt.show()
print("10-bit:", 84e6 / 2 ** 10, "Hz")
print("1 s from 72 MHz:", it.find_psc_arr(72e6, 1.0))
print("watchdog 64 x 1000 at 24/32/40 kHz:", [round(it.watchdog_timeout(64, 1000, f), 3) for f in (24e3, 32e3, 40e3)])

10-bit: 82031.25 Hz
1 s from 72 MHz: (1124, 63999)
watchdog 64 x 1000 at 24/32/40 kHz: [2.667, 2.0, 1.6]


## Discussion

The questions and worked answers are in the [handout](README.md#discussion).